## IAA-Transformer — Imbalance-Aware Attention Transformer

### What It Does
IAA-Transformer solves class imbalance at the architecture level, not the data level. Instead of generating synthetic samples or reweighting losses, it modifies the attention mechanism itself to force the model to pay more attention to rare attack classes during every forward pass.

### The Formula
IAA(Q, K, V) = softmax(QK^T / √dk + α · log(1/f_c)) · V
The term `α · log(1/f_c)` is a class frequency bias added directly to the attention scores. Rare classes have small `f_c`, which makes `log(1/f_c)` large, which forces higher attention scores for those classes. When `α = 0`, the formula collapses to standard Transformer attention — used as the ablation study baseline.

### Why Custom Implementation
PyTorch's built-in `nn.TransformerEncoderLayer` does not allow modifying the internal attention formula. We implement our own attention class from scratch so we can inject `α · log(1/f_c)` directly into the score calculation before softmax.

In [2]:
#IAAA
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import math

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [3]:
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = pd.read_csv('../data/processed/y_train_resampled.csv').squeeze()
X_val = np.load('../data/processed/X_val_scaled.npy')
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (2062829, 71)
X_val shape: (423051, 71)


In [4]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# Calculate class frequencies from training data
total_samples = len(y_train)
class_freq = {}
for label in le.classes_:
    count = (y_train == label).sum()
    class_freq[label] = count / total_samples

print("Class frequencies:")
for label, freq in class_freq.items():
    print(f"  {label}: {freq:.6f} → log(1/f) = {math.log(1/freq):.4f}")

Class frequencies:
  BENIGN: 0.771153 → log(1/f) = 0.2599
  Bot: 0.004848 → log(1/f) = 5.3292
  DDoS: 0.043468 → log(1/f) = 3.1357
  DoS GoldenEye: 0.004848 → log(1/f) = 5.3292
  DoS Hulk: 0.078134 → log(1/f) = 2.5493
  DoS Slowhttptest: 0.004848 → log(1/f) = 5.3292
  DoS slowloris: 0.004848 → log(1/f) = 5.3292
  FTP-Patator: 0.004848 → log(1/f) = 5.3292
  Heartbleed: 0.004848 → log(1/f) = 5.3292
  Infiltration: 0.004848 → log(1/f) = 5.3292
  PortScan: 0.053919 → log(1/f) = 2.9203
  SSH-Patator: 0.004848 → log(1/f) = 5.3292
  Web Attack - Brute Force: 0.004848 → log(1/f) = 5.3292
  Web Attack - Sql Injection: 0.004848 → log(1/f) = 5.3292
  Web Attack - XSS: 0.004848 → log(1/f) = 5.3292


In [5]:
# Load original y_train before SMOTE for true class frequencies
y_train_original = pd.read_csv('../data/processed/y_train_clean.csv').squeeze()

total_original = len(y_train_original)
class_freq_original = {}
for label in le.classes_:
    count = (y_train_original == label).sum()
    class_freq_original[label] = count / total_original

print("Original class frequencies (before SMOTE):")
for label, freq in class_freq_original.items():
    print(f"  {label}: {freq:.8f} → log(1/f) = {math.log(1/freq):.4f}")

Original class frequencies (before SMOTE):
  BENIGN: 0.80318222 → log(1/f) = 0.2192
  Bot: 0.00069172 → log(1/f) = 7.2763
  DDoS: 0.04527388 → log(1/f) = 3.0950
  DoS GoldenEye: 0.00363986 → log(1/f) = 5.6158
  DoS Hulk: 0.08137969 → log(1/f) = 2.5086
  DoS Slowhttptest: 0.00194439 → log(1/f) = 6.2428
  DoS slowloris: 0.00204992 → log(1/f) = 6.1900
  FTP-Patator: 0.00280627 → log(1/f) = 5.8759
  Heartbleed: 0.00000353 → log(1/f) = 12.5530
  Infiltration: 0.00001313 → log(1/f) = 11.2408
  PortScan: 0.05615864 → log(1/f) = 2.8796
  SSH-Patator: 0.00208526 → log(1/f) = 6.1729
  Web Attack - Brute Force: 0.00053318 → log(1/f) = 7.5367
  Web Attack - Sql Injection: 0.00000757 → log(1/f) = 11.7908
  Web Attack - XSS: 0.00023074 → log(1/f) = 8.3742


In [6]:
# Create frequency bias lookup tensor
freq_bias = torch.zeros(len(le.classes_))
for i, label in enumerate(le.classes_):
    freq_bias[i] = math.log(1.0 / class_freq[label])

print("Frequency bias tensor:")
for i, label in enumerate(le.classes_):
    print(f"  {i} ({label}): {freq_bias[i]:.4f}")

Frequency bias tensor:
  0 (BENIGN): 0.2599
  1 (Bot): 5.3292
  2 (DDoS): 3.1357
  3 (DoS GoldenEye): 5.3292
  4 (DoS Hulk): 2.5493
  5 (DoS Slowhttptest): 5.3292
  6 (DoS slowloris): 5.3292
  7 (FTP-Patator): 5.3292
  8 (Heartbleed): 5.3292
  9 (Infiltration): 5.3292
  10 (PortScan): 2.9203
  11 (SSH-Patator): 5.3292
  12 (Web Attack - Brute Force): 5.3292
  13 (Web Attack - Sql Injection): 5.3292
  14 (Web Attack - XSS): 5.3292


In [7]:
# Create frequency bias lookup tensor
freq_bias = torch.zeros(len(le.classes_))
for i, label in enumerate(le.classes_):
    freq_bias[i] = math.log(1.0 / class_freq[label])

print("Frequency bias tensor:")
for i, label in enumerate(le.classes_):
    print(f"  {i} ({label}): {freq_bias[i]:.4f}")

Frequency bias tensor:
  0 (BENIGN): 0.2599
  1 (Bot): 5.3292
  2 (DDoS): 3.1357
  3 (DoS GoldenEye): 5.3292
  4 (DoS Hulk): 2.5493
  5 (DoS Slowhttptest): 5.3292
  6 (DoS slowloris): 5.3292
  7 (FTP-Patator): 5.3292
  8 (Heartbleed): 5.3292
  9 (Infiltration): 5.3292
  10 (PortScan): 2.9203
  11 (SSH-Patator): 5.3292
  12 (Web Attack - Brute Force): 5.3292
  13 (Web Attack - Sql Injection): 5.3292
  14 (Web Attack - XSS): 5.3292


In [8]:
class NetworkFlowDataset(Dataset):
    def __init__(self, X, y, freq_bias):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.freq_bias = freq_bias
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        label = self.y[idx]
        bias = self.freq_bias[label]
        return self.X[idx], label, bias

train_dataset = NetworkFlowDataset(X_train, y_train_encoded, freq_bias)
val_dataset = NetworkFlowDataset(X_val, y_val_encoded, freq_bias)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 4029
Validation batches: 827


In [9]:
class IAAAttention(nn.Module):
    def __init__(self, d_model, nhead):
        super(IAAAttention, self).__init__()
        
        self.d_model = d_model
        self.nhead = nhead
        self.d_head = d_model // nhead
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.alpha = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, x, class_bias):
        batch_size, seq_len, _ = x.shape
        
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        Q = Q.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.nhead, self.d_head).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        bias = self.alpha * class_bias.view(batch_size, 1, 1, 1)
        scores = scores + bias
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(output)
        
        return output

In [10]:
#full model
class IAATransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=512, dropout=0.3):
        super(IAATransformerLayer, self).__init__()
        
        self.attention = IAAAttention(d_model, nhead)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, class_bias):
        # Attention + residual connection
        attn_output = self.attention(x, class_bias)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed forward + residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x


class IAATransformer(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, num_classes, dropout=0.3):
        super(IAATransformer, self).__init__()
        
        self.input_projection = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([
            IAATransformerLayer(d_model, nhead, dim_feedforward=512, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, class_bias):
        x = x.unsqueeze(2)
        x = self.input_projection(x)
        
        for layer in self.layers:
            x = layer(x, class_bias)
        
        x = x.mean(dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [11]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = IAATransformer(
    input_size=71,
    d_model=128,
    nhead=4,
    num_layers=3,
    num_classes=15,
    dropout=0.3
).to(device)

freq_bias = freq_bias.to(device)

print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps
Parameters: 597,010


In [12]:
#loss function
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

print("Loss: CrossEntropyLoss")
print("Optimizer: Adam, lr=0.0005")

Loss: CrossEntropyLoss
Optimizer: Adam, lr=0.0005


In [13]:
#training loop
def train_model(model, train_loader, criterion, optimizer, epochs=20):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_y, batch_bias in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            batch_bias = batch_bias.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X, batch_bias)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

train_model(model, train_loader, criterion, optimizer, epochs=20)

TypeError: IAATransformerLayer.forward() missing 1 required positional argument: 'class_bias'

In [15]:
torch.save(model.state_dict(), '../models/transformer.pth')
print("Model saved.")

Model saved.


In [18]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y, batch_bias in val_loader:
        batch_X = batch_X.to(device)
        batch_bias = batch_bias.to(device)
        
        outputs = model(batch_X, batch_bias)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds = le.inverse_transform(all_preds)
all_labels = le.inverse_transform(all_labels)

print(classification_report(all_labels, all_preds, digits=4))

                            precision    recall  f1-score   support

                    BENIGN     0.9690    0.9663    0.9676    339790
                       Bot     0.3650    0.3276    0.3453       293
                      DDoS     0.9945    0.7825    0.8759     19153
             DoS GoldenEye     0.9208    0.8156    0.8650      1540
                  DoS Hulk     0.8339    0.8579    0.8457     34427
          DoS Slowhttptest     0.7423    0.9380    0.8288       823
             DoS slowloris     0.9790    0.8051    0.8835       867
               FTP-Patator     0.9304    0.9798    0.9545      1187
                Heartbleed     0.0046    1.0000    0.0092         2
              Infiltration     0.0077    0.6000    0.0152         5
                  PortScan     0.8871    0.9534    0.9190     23757
               SSH-Patator     0.7105    0.7653    0.7369       882
  Web Attack - Brute Force     0.3704    0.0889    0.1434       225
Web Attack - Sql Injection     0.0020    0.3333